# AMD/NVIDIA 原始季度公告事件身份审查

复用 133 份 SEC 原件和 11 份未匹配复核；排除初步业绩公告，保留定性指引缺口及后发布指引的真实时点。

In [1]:
import json,hashlib
from pathlib import Path
from collections import Counter
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'quarter_event_adjudication_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'item202_inventory_result.json')==plan['parent_inventory_result_sha256'] and sha(card/'item202_nonmatch_repair_result.json')==plan['parent_nonmatch_repair_result_sha256'] and not (card/'quarter_event_adjudication_result.json').exists()
a=json.loads((card/'item202_inventory_result.json').read_text());b=json.loads((card/'item202_nonmatch_repair_result.json').read_text());assert a['NAS_archive_sha256']==plan['parent_NAS_archive_sha256'] and a['NAS_remote_and_scoped_restore_verified'] and a['processed']==133
recovered={x['accession'] for x in b['rows'] if x['classification'].startswith('numeric_forward')};prelim=set(plan['exact_initial_false_positive_preliminary_accessions']);qual=plan['AMD_qualitative_no_numeric_outlook_accession'];prior=plan['NVDA_prior_earnings_no_same_release_numeric_outlook_accession']
source_by={x['accession']:x for x in a['checks']};assert all(source_by[x]['original_quarter_results_and_forward_revenue_outlook'] for x in prelim) and not source_by[qual]['original_quarter_results_and_forward_revenue_outlook'] and not source_by[prior]['original_quarter_results_and_forward_revenue_outlook']
slot_months={'AMD':{'01':'Q4_prev','02':'Q4_prev','04':'Q1','05':'Q1','07':'Q2','08':'Q2','10':'Q3','11':'Q3'},'NVDA':{'02':'Q4','05':'Q1','08':'Q2','11':'Q3'}}
selected=[]
for x in a['checks']:
 acc=x['accession'];issuer=x['issuer'];numeric=(x['original_quarter_results_and_forward_revenue_outlook'] or acc in recovered) and acc not in prelim;qualitative=acc==qual
 if not (numeric or qualitative):continue
 day=x['filed'];slot=slot_months[issuer].get(day[5:7]);assert slot,(issuer,day,acc)
 p=raw/f'item202_{acc}.txt';assert sha(p)==x['raw_sha256']
 selected.append({'issuer':issuer,'announcement_year':int(day[:4]),'slot':slot,'filing_date':day,'accession':acc,'raw_sha256':x['raw_sha256'],'revenue_outlook':'numeric' if numeric else 'qualitative_unknown','source':'initial_item202' if x['original_quarter_results_and_forward_revenue_outlook'] else 'saved_nonmatch_repair'})
selected.sort(key=lambda x:(x['issuer'],x['announcement_year'],x['slot'],x['filing_date'],x['accession']))
keys=[(x['issuer'],x['announcement_year'],x['slot']) for x in selected];assert len(keys)==len(set(keys))==126
by={issuer:Counter(x['revenue_outlook'] for x in selected if x['issuer']==issuer) for issuer in ('AMD','NVDA')};assert by['AMD']=={'numeric':62,'qualitative_unknown':1} and by['NVDA']=={'numeric':63}
for issuer in ('AMD','NVDA'):
 own=[x for x in selected if x['issuer']==issuer];assert len(own)==63
 for year in range(2011,2026):assert sum(x['announcement_year']==year for x in own)==4,(issuer,year)
 assert sum(x['announcement_year']==2026 for x in own)==3
result={'status':'AMD_NVDA_original_quarter_event_slots_126_identity_complete_one_qualitative_numeric_gap','plan_sha256':sha(card/'quarter_event_adjudication_plan.json'),'parent_inventory_result_sha256':plan['parent_inventory_result_sha256'],'parent_NAS_archive_sha256':plan['parent_NAS_archive_sha256'],'selected_events':selected,'selected_slots_each_issuer':63,'numeric_forward_guidance_each_issuer':{'AMD':62,'NVDA':63},'qualitative_early_AMD_accession':qual,'excluded_preliminary_accessions':sorted(prelim),'NVDA_2014_earlier_earnings_without_outlook_accession':prior,'earlier_NVDA_release_not_backfilled':True,'new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'numeric_harmonization_qualified':False,'first_public_issuer_clock_qualified':False};(card/'quarter_event_adjudication_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'events':len(selected),'numeric':result['numeric_forward_guidance_each_issuer'],'one_qualitative':qual,'new_GETs':0})


{'status': 'AMD_NVDA_original_quarter_event_slots_126_identity_complete_one_qualitative_numeric_gap', 'events': 126, 'numeric': {'AMD': 62, 'NVDA': 63}, 'one_qualitative': '0001193125-11-011350', 'new_GETs': 0}


## 中文结论

排除初筛误收的 3 份初步公告后，两家公司各 63 个季度事件时段；AMD 一期定性未知，NVIDIA 2014 年后发指引不回填。 完整决定见 `conclusion.md`。